# CampusGPT: QLoRA fine-tuning on Colab (free T4)
Runtime → Change runtime type → **T4 GPU**. Upload `campusgpt.zip` when asked.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall('.')
os.chdir('campusgpt')
!pip -q install -r requirements-train.txt

In [ ]:
# Rebuild chunks from the PDFs (already included, but safe to re-run)
!python -m campusgpt.ingest

In [ ]:
# Generate the QA dataset with Claude (needs an API key). Skip this cell if data/qa/train.jsonl already exists.
import getpass, os
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')
!python -m training.make_qa_dataset --n 80 --backend anthropic

In [ ]:
# SPOT-CHECK about 50 pairs by hand before training
import pandas as pd
pd.read_json('data/qa/qa_raw.jsonl', lines=True).sample(10, random_state=0)

In [ ]:
# Train (about 20-60 min on a T4). If out of memory: --batch 1 --max-len 1024, or BASE_MODEL=Qwen/Qwen2.5-0.5B-Instruct
!python -m training.train_qlora --epochs 2

In [ ]:
from IPython.display import Image
Image('training/outputs/campusgpt-lora/loss_curve.png')

In [ ]:
# Evaluate all four setups (Claude judges correctness / hallucination / faithfulness)
!python -m evaluation.evaluate --configs base,base+rag,ft,ft+rag --judge

In [ ]:
# Download the adapter and results
!zip -qr results.zip training/outputs/campusgpt-lora data/qa data/eval
files.download('results.zip')